In [1]:
import os
import pandas as pd
from datetime import datetime
import pytz
import unicodedata

# === Date & Paths ===
eastern = pytz.timezone("US/Eastern")
today = datetime.now(eastern).date()
today_str = today.isoformat()

paths = {
    "spreads": f"../data/novig-odds/novig_spreads_{today_str}.csv",
    "pitching_stats": f"../data/pitching_stats/pitching_stats_{today_str}.csv",
    "standings": f"../data/standings/standings_{today_str}.csv",
    "team_batting": f"../data/team_batting/team_batting_{today_str}.csv",
    "team_pitching": f"../data/team_pitching/team_pitching_{today_str}.csv"
}

# === Load Data ===
for label, path in paths.items():
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing required file: {path}")

spreads_df = pd.read_csv(paths["spreads"])
pitching_stats_df = pd.read_csv(paths["pitching_stats"])
standings_df = pd.read_csv(paths["standings"])
batting_df = pd.read_csv(paths["team_batting"])
teamp_df = pd.read_csv(paths["team_pitching"])

# === Normalize ===
def normalize_str(s):
    if not isinstance(s, str): return ""
    return unicodedata.normalize('NFKD', s).encode('ascii', 'ignore').decode('utf-8').strip().lower()

# === Preprocess Spreads ===
spreads_df['fav_score'] = None
spreads_df['dog_score'] = None
spreads_df['home_score'] = None
spreads_df['away_score'] = None

# === Normalize team names ===
pitching_stats_df['Home Team Clean'] = pitching_stats_df['Home Team'].apply(normalize_str)
pitching_stats_df['Away Team Clean'] = pitching_stats_df['Away Team'].apply(normalize_str)

# === Map team abbreviations to full names ===
team_name_map = {
    "ATL": "Atlanta Braves", "PHI": "Philadelphia Phillies", "OAK": "Athletics",
    "TOR": "Toronto Blue Jays", "TB": "Tampa Bay Rays", "HOU": "Houston Astros",
    "WAS": "Washington Nationals", "SEA": "Seattle Mariners", "CIN": "Cincinnati Reds",
    "CHC": "Chicago Cubs", "MIL": "Milwaukee Brewers", "CWS": "Chicago White Sox",
    "BAL": "Baltimore Orioles", "LAA": "Los Angeles Angels", "CLE": "Cleveland Guardians",
    "COL": "Colorado Rockies", "NYM": "New York Mets", "SF": "San Francisco Giants",
    "MIA": "Miami Marlins", "BOS": "Boston Red Sox", "STL": "St. Louis Cardinals",
    "TEX": "Texas Rangers", "DET": "Detroit Tigers", "KC": "Kansas City Royals",
    "KAN": "Kansas City Royals", "AZ": "Arizona Diamondbacks", "ARI": "Arizona Diamondbacks",
    "PIT": "Pittsburgh Pirates", "SD": "San Diego Padres", "MIN": "Minnesota Twins",
    "NYY": "New York Yankees", "LAD": "Los Angeles Dodgers"
}

# Derive home and away teams
spreads_df['home_team'] = spreads_df.apply(lambda row: row['fav_team'] if row['home_favorite'] == 1 else row['dog_team'], axis=1)
spreads_df['away_team'] = spreads_df.apply(lambda row: row['dog_team'] if row['home_favorite'] == 1 else row['fav_team'], axis=1)
spreads_df['home_team_full'] = spreads_df['home_team'].map(lambda abbr: team_name_map.get(abbr.upper(), ""))
spreads_df['away_team_full'] = spreads_df['away_team'].map(lambda abbr: team_name_map.get(abbr.upper(), ""))
spreads_df['home_team_clean'] = spreads_df['home_team_full'].apply(normalize_str)
spreads_df['away_team_clean'] = spreads_df['away_team_full'].apply(normalize_str)

# === Assign Pitchers and Stats ===
output_rows = []

for _, row in spreads_df.iterrows():
    home = row['home_team_clean']
    away = row['away_team_clean']

    ps_row = pitching_stats_df[
        (pitching_stats_df['Home Team Clean'] == home) &
        (pitching_stats_df['Away Team Clean'] == away)
    ]

    if ps_row.empty:
        ps_row = pitching_stats_df[
            (pitching_stats_df['Home Team Clean'] == away) &
            (pitching_stats_df['Away Team Clean'] == home)
        ]
        flipped = not ps_row.empty
    else:
        flipped = False

    if not ps_row.empty:
        ps_row = ps_row.iloc[0]
        if flipped:
            home_pitcher, away_pitcher = ps_row['Away Starter'], ps_row['Home Starter']
            home_stats = {
                'era': ps_row['Away ERA'], 'whip': ps_row['Away WHIP'], 'so': ps_row['Away SO'],
                'ip': ps_row['Away IP'], 'avg': ps_row['Away AVG']
            }
            away_stats = {
                'era': ps_row['Home ERA'], 'whip': ps_row['Home WHIP'], 'so': ps_row['Home SO'],
                'ip': ps_row['Home IP'], 'avg': ps_row['Home AVG']
            }
        else:
            home_pitcher, away_pitcher = ps_row['Home Starter'], ps_row['Away Starter']
            home_stats = {
                'era': ps_row['Home ERA'], 'whip': ps_row['Home WHIP'], 'so': ps_row['Home SO'],
                'ip': ps_row['Home IP'], 'avg': ps_row['Home AVG']
            }
            away_stats = {
                'era': ps_row['Away ERA'], 'whip': ps_row['Away WHIP'], 'so': ps_row['Away SO'],
                'ip': ps_row['Away IP'], 'avg': ps_row['Away AVG']
            }

        if row['home_favorite'] == 1:
            fav_pitcher, dog_pitcher = home_pitcher, away_pitcher
            fav_stats, dog_stats = home_stats, away_stats
        else:
            fav_pitcher, dog_pitcher = away_pitcher, home_pitcher
            fav_stats, dog_stats = away_stats, home_stats
    else:
        fav_pitcher = dog_pitcher = "Unknown"
        fav_stats = dog_stats = {k: None for k in ['era', 'whip', 'so', 'ip', 'avg']}

    row = row.copy()
    row['fav_pitcher'] = fav_pitcher
    row['dog_pitcher'] = dog_pitcher
    for k in fav_stats:
        row[f'fav_pitcher_{k}'] = fav_stats[k]
        row[f'dog_pitcher_{k}'] = dog_stats[k]

    output_rows.append(row)

merged_df = pd.DataFrame(output_rows)

# === Normalize and Join Team Stats ===
for df in [standings_df, batting_df, teamp_df]:
    df['team_clean'] = df['Tm'].apply(normalize_str)

def get_team_stat(team_abbr, df, column):
    full = team_name_map.get(team_abbr.upper(), "")
    clean = normalize_str(full)
    row = df[df['team_clean'] == clean]
    return row[column].values[0] if not row.empty else None

# Standings
merged_df['fav_win_pct'] = merged_df['fav_team'].apply(lambda abbr: get_team_stat(abbr, standings_df, 'W-L%'))
merged_df['dog_win_pct'] = merged_df['dog_team'].apply(lambda abbr: get_team_stat(abbr, standings_df, 'W-L%'))

# Batting/ERA
merged_df['fav_ba'] = merged_df['fav_team'].apply(lambda abbr: get_team_stat(abbr, batting_df, 'BA'))
merged_df['dog_ba'] = merged_df['dog_team'].apply(lambda abbr: get_team_stat(abbr, batting_df, 'BA'))
merged_df['fav_era_p'] = merged_df['fav_team'].apply(lambda abbr: get_team_stat(abbr, teamp_df, 'ERA'))
merged_df['dog_era_p'] = merged_df['dog_team'].apply(lambda abbr: get_team_stat(abbr, teamp_df, 'ERA'))

# Additional standings columns
standings_columns = ['W','L','Strk','R','RA','SOS','SRS','pythWL','Luck','last10','last20','last30']
for col in standings_columns:
    merged_df[f'fav_{col}'] = merged_df['fav_team'].apply(lambda abbr: get_team_stat(abbr, standings_df, col))
    merged_df[f'dog_{col}'] = merged_df['dog_team'].apply(lambda abbr: get_team_stat(abbr, standings_df, col))

# Additional batting columns
batting_columns = ['R/G','PA','AB','R','H','2B','3B','HR','RBI','SB','BB','SO','BA','OBP','SLG','OPS','OPS+']
for col in batting_columns:
    col_renamed = f'{col}_b' if col in ['R','H','BB','SO'] else col
    merged_df[f'fav_{col_renamed}'] = merged_df['fav_team'].apply(lambda abbr: get_team_stat(abbr, batting_df, col))
    merged_df[f'dog_{col_renamed}'] = merged_df['dog_team'].apply(lambda abbr: get_team_stat(abbr, batting_df, col))

# Additional pitching columns
pitching_columns = ['RA/G','ERA','SV','H','R','ER','HR','BB','SO','ERA+','FIP','WHIP','H9']
for col in pitching_columns:
    col_renamed = f'{col}_p' if col in ['H','R','BB','SO'] else col
    merged_df[f'fav_{col_renamed}'] = merged_df['fav_team'].apply(lambda abbr: get_team_stat(abbr, teamp_df, col))
    merged_df[f'dog_{col_renamed}'] = merged_df['dog_team'].apply(lambda abbr: get_team_stat(abbr, teamp_df, col))

# === Final Ordering ===
first_cols = [
    'game_time_est', 'fav_team', 'fav_score', 'dog_team', 'dog_score',
    'home_team', 'home_score', 'away_team', 'away_score',
    'fav_line', 'dog_line', 'fav_price', 'dog_price',
    'fav_price_american', 'dog_price_american', 'home_favorite', 'market',
    'fav_pitcher', 'dog_pitcher',
    'fav_pitcher_era', 'dog_pitcher_era',
    'fav_pitcher_whip', 'dog_pitcher_whip',
    'fav_pitcher_so', 'dog_pitcher_so',
    'fav_pitcher_ip', 'dog_pitcher_ip',
    'fav_pitcher_avg', 'dog_pitcher_avg',
    'fav_win_pct', 'dog_win_pct', 'fav_ba', 'dog_ba', 'fav_era_p', 'dog_era_p'
]
remaining_cols = [col for col in merged_df.columns if col not in first_cols and col not in ['market_timestamp', 'home_team_full', 'away_team_full', 'home_team_clean', 'away_team_clean']]
merged_df = merged_df[first_cols + remaining_cols]

# === Output ===
print("\u2705 Final enriched training set (WAR excluded):")
pd.set_option('display.max_columns', None)
pd.set_option('display.expand_frame_repr', False)
display(merged_df)

# === Save CSV ===
output_path = f"../training-data/training-set/training_set_{today_str}.csv"
os.makedirs(os.path.dirname(output_path), exist_ok=True)
merged_df.to_csv(output_path, index=False)
print(f"\u2705 Enriched training set saved to {output_path}")


✅ Final enriched training set (WAR excluded):


,game_time_est,fav_team,fav_score,dog_team,dog_score,home_team,home_score,away_team,away_score,fav_line,dog_line,fav_price,dog_price,fav_price_american,dog_price_american,home_favorite,market,fav_pitcher,dog_pitcher,fav_pitcher_era,dog_pitcher_era,fav_pitcher_whip,dog_pitcher_whip,fav_pitcher_so,dog_pitcher_so,fav_pitcher_ip,dog_pitcher_ip,fav_pitcher_avg,dog_pitcher_avg,fav_win_pct,dog_win_pct,fav_ba,dog_ba,fav_era_p,dog_era_p,fav_W,dog_W,fav_L,dog_L,fav_Strk,dog_Strk,fav_R,dog_R,fav_RA,dog_RA,fav_SOS,dog_SOS,fav_SRS,dog_SRS,fav_pythWL,dog_pythWL,fav_Luck,dog_Luck,fav_last10,dog_last10,fav_last20,dog_last20,fav_last30,dog_last30,fav_R/G,dog_R/G,fav_PA,dog_PA,fav_AB,dog_AB,fav_R_b,dog_R_b,fav_H_b,dog_H_b,fav_2B,dog_2B,fav_3B,dog_3B,fav_HR,dog_HR,fav_RBI,dog_RBI,fav_SB,dog_SB,fav_BB_b,dog_BB_b,fav_SO_b,dog_SO_b,fav_BA,dog_BA,fav_OBP,dog_OBP,fav_SLG,dog_SLG,fav_OPS,dog_OPS,fav_OPS+,dog_OPS+,fav_RA/G,dog_RA/G,fav_ERA,dog_ERA,fav_SV,dog_SV,fav_H_p,dog_H_p,fav_R_p,dog_R_p,fav_ER,dog_ER,fav_BB_p,dog_BB_p,fav_SO_p,dog_SO_p,fav_ERA+,dog_ERA+,fav_FIP,dog_FIP,fav_WHIP,dog_WHIP,fav_H9,dog_H9
0,2025-06-04T19:07:00-04:00,TOR,None,PHI,None,TOR,None,PHI,None,-1.5,1.5,0.342,0.679,192,-212,1,TOR -1.5,José Berríos,Mick Abel,3.86,0.00,1.30,0.83,65.0,9.0,70.0,6.0,0.242,0.227,0.517,0.617,.253,.257,4.08,3.99,31,37,29,23,L 1,W 1,4.2,4.8,4.4,4.4,-0.1,-0.2,-0.2,0.2,29-31,33-27,2.0,4.0,6-4,5-5,11-9,13-7,17-13,20-10,4.18,4.82,2281,2317,2014,2054,251,289,509,528,102,92,2,9,84,62,241,274,35,56,211,221,420,473,.253,.257,.327,.333,.395,.407,.722,.740,103,105,4.35,4.37,4.08,3.99,18,20,477,520,261,262,243,239,175,187,547,578,103,104,4.20,3.56,1.217,1.310,8.0,8.7
1,2025-06-04T19:15:00-04:00,ATL,None,ARI,None,ATL,None,ARI,None,-1.5,1.5,0.435,0.582,130,-139,1,ATL -1.5,Chris Sale,Merrill Kelly,3.06,3.78,1.24,1.06,86.0,64.0,67.2,69.0,0.252,0.212,0.458,0.483,.245,.255,3.73,4.75,27,29,32,31,L 2,W 2,4.1,5.1,3.9,5.2,0.0,0.3,0.1,0.2,30-29,30-30,-3.0,-1.0,3-7,3-7,8-12,8-12,13-17,13-17,4.05,5.07,2236,2315,2002,2038,239,304,491,520,82,117,6,12,67,75,229,297,35,51,200,216,501,446,.245,.255,.317,.332,.387,.448,.704,.780,97,115,3.92,5.15,3.73,4.75,10,18,456,511,231,309,216,282,192,199,514,507,110,88,4.01,4.27,1.244,1.328,7.9,8.6
2,2025-06-04T18:40:00-04:00,HOU,None,PIT,None,PIT,None,HOU,None,-1.5,1.5,0.433,0.597,131,-148,0,PIT +1.5,Ryan Gusto,Mike Burrows,4.62,8.64,1.56,1.56,42.0,5.0,39.0,8.1,0.268,0.265,0.550,0.361,.252,.226,3.59,3.94,33,22,27,39,W 2,L 2,4.0,3.2,3.7,4.2,0.1,0.3,0.4,-0.7,32-28,23-38,1.0,-1.0,7-3,5-5,13-7,8-12,17-13,10-20,4.03,3.18,2215,2275,1991,2026,242,194,502,458,81,75,6,10,59,50,231,188,30,59,173,209,457,533,.252,.226,.319,.304,.391,.337,.709,.641,100,79,3.73,4.18,3.59,3.94,17,12,427,478,224,255,211,237,186,191,569,444,113,107,3.54,3.83,1.160,1.237,7.3,8.0
3,2025-06-04T19:35:00-04:00,TB,None,TEX,None,TEX,None,TB,None,-1.5,1.5,0.376,0.656,166,-191,0,TB -1.5,Shane Baz,Kumar Rocker,4.92,8.10,1.38,1.75,55.0,16.0,60.1,20.0,0.265,0.341,0.517,0.475,.246,.222,3.44,3.13,31,29,29,32,W 1,L 1,4.4,3.4,3.6,3.4,0.0,0.0,0.8,0.0,35-25,30-31,-4.0,-1.0,7-3,4-6,13-7,9-11,17-13,13-17,4.35,3.36,2221,2159,2003,1963,261,205,493,435,93,71,3,6,78,52,249,200,84,55,182,156,502,496,.246,.222,.312,.284,.388,.356,.700,.639,100,85,3.60,3.38,3.44,3.13,14,17,461,436,216,206,205,185,159,174,464,471,113,120,4.25,3.65,1.156,1.148,7.7,7.4
4,2025-06-04T21:45:00-04:00,SD,None,SF,None,SF,None,SD,None,-1.5,1.5,0.388,0.634,158,-173,0,SF +1.5,Nick Pivetta,Kyle Harrison,2.74,2.51,1.01,0.98,71.0,16.0,62.1,14.1,0.199,0.160,0.593,0.541,.248,.231,3.51,3.03,35,33,24,28,W 3,L 2,4.2,4.1,3.9,3.4,-0.2,-0.1,0.2,0.6,32-27,35-26,3.0,-2.0,7-3,3-7,10-10,9-11,17-13,14-16,4.17,4.10,2191,2271,1958,2014,246,250,486,465,86,88,7,9,61,41,226,239,43,36,184,213,409,515,.248,.231,.315,.309,.380,.370,.695,.679,95,95,3.86,3.44,3.51,3.03,22,17,441,460,228,210,205,181,190,184,517,529,115,126,3.83,3.24,1.200,1.197,7.5,7.7
5,2025-06-04T19:45:00-04:00,STL,None,KAN,None,KAN,None,STL,None,-1.5,1.5,0.377,0.659,165,-193,0,STL -1.5,M

✅ Enriched training set saved to ../training-data/training-set/training_set_2025-06-04.csv
